# 🏛️ Rajat Alpha V67: Master Quantitative Trading Suite & Mission Control
**Author**: Senior Quantitative Strategist & Multi-Time US Investing Champion (15+ Years Multi-Cycle Experience)  
**Target Engine**: `Single_Buy2.0` / `rajat_alpha_v67_single.py` Live & Paper Trading Engine  
**Version**: 3.5.1 Enterprise Unified Platform

---

## 🎯 Executive Mission Control Overview
Welcome to the **Master Trading Suite** for the Rajat Alpha V67 institutional trading ecosystem. This notebook serves as the **central landing page and unified orchestration platform** that bridges the entire lifecycle of systematic swing trading:

```mermaid
flowchart TD
    Universe["US Market Universe (~850+ Equities)
(Indices + Broader Growth & Mid-Caps)"] --> M1["MODULE 1: Weekly Swing Radar & Screener
(13 Trend Rules, Slopes, Demand Zones, A+ to B- Grades)"]
    M1 -->|Auto Pipeline Feed: Top 50 Candidates| M2["MODULE 2: Multi-Year Vectorized Backtester
(Fixed 3% vs Risk-Based Equal Risk, 4-Tier Exits)"]
    M2 -->|Auto Pipeline Feed: Historical Trade Log & Metrics| M3["MODULE 3: Institutional Prop Protocol & Monte Carlo
(10,000 Simulation Runs, Max Drawdown Stress Test)"]
    M1 -->|Export: watchlist.txt| M4["MODULE 4: Live Execution Engine
(Single_Buy2.0 / rajat_alpha_v67_single.py)"]
```

---

## 📂 Modular Sub-Colab Directory Architecture
Each module in this ecosystem is self-contained in its own dedicated subfolder under `scripts/colab/` and **can be executed 100% independently**, or run seamlessly end-to-end directly inside this Master Notebook:

| Directory | Sub-Notebook | Primary Mission | Standalone Input / Output |
|---|---|---|---|
| [`01_weekly_screener/`](./01_weekly_screener/) | `Rajat_Alpha_Institutional_Weekly_Screener.ipynb` | **Weekly Candidate Discovery**: Scans ~850 tickers, evaluates 21 EMA/50 SMA buy zones, daily/weekly demand, pocket pivots, swing geometry ($SL_1, SL_2, SH_1, SH_2$), grades setups $A+$ to $B-$. | **Out**: `weekly_swing_screener_results.xlsx` (6 sheets), `watchlist.txt` (top 50) |
| [`02_swing_backtest/`](./02_swing_backtest/) | `Rajat_Alpha_v67_Colab_Backtest.ipynb` | **Strategy Backtest Engine**: 5-year historical simulation evaluating Gate 1-6 entries, 4-tier partial exits, and comparing Fixed 3% Capital Allocation vs Risk-Based Equal Risk Contribution. | **In**: Custom watchlist or screener output<br>**Out**: Trade logs, equity curves, Sharpe/DD |
| [`03_institutional_prop_protocol/`](./03_institutional_prop_protocol/) | `Rajat_Alpha_Institutional_Prop_Protocol.ipynb` | **Prop Firm Risk & Monte Carlo**: 4-step framework (Map, Measure, Simulate, Operate), 10,000-run Monte Carlo simulation, 4 risk profiles (Conservative, Target, Aggressive, Ultra). | **In**: Strategy win rate & expectancy<br>**Out**: `rajat_alpha_4_configs_master_simulation.xlsx` |
| [`04_scanned_watchlist_backtest/`](./04_scanned_watchlist_backtest/) | `Rajat_Alpha_v67_Scanned_Watchlist_Backtest.ipynb` | **Scanned Watchlist Multi-Year Validation**: Backtests user weekly watchlists against historical impulse & pullback patterns. | **In**: `watchlist.txt`<br>**Out**: Symbol rankings & suitability tiers |
| [`05_reference_simulations/`](./05_reference_simulations/) | `TheInstitutionalProtocol_PropSimulations.ipynb` | **Foundational Institutional Reference**: Original masterclass protocol reference notebooks. | **Out**: Analytical prop distributions |

---

## ⚡ How the Automated Pipeline Handshake Works
When running this Master Notebook:
1. **Module 1 (Screener)** runs first and produces the top candidates rated $A+$ to $B-$.
2. The candidates are **automatically passed in-memory** to **Module 2 (Backtester)** as `PIPELINE_WATCHLIST` without any manual copy-pasting.
3. Module 2 simulates trades across the screened candidates and pipes the empirical win rate and trade log to **Module 3 (Monte Carlo)**.
4. You can also toggle the Pipeline Mode to run **any module independently**!


In [ ]:
# @title 📦 Step 1: Install Dependencies & Import Institutional Libraries
# @markdown Run this cell once to configure the Google Colab Python 3 environment:

import sys
import subprocess
import os

required_pkgs = ['yfinance', 'pandas', 'numpy', 'matplotlib', 'openpyxl', 'scipy', 'ipywidgets']
installed_pkgs = []

for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f"📦 Installing required library: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        installed_pkgs.append(pkg)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Environment initialized successfully with Python 3 kernel.")
print("📊 All quantitative libraries loaded: yfinance, pandas, numpy, matplotlib, openpyxl, ipywidgets.")


In [ ]:
# @title ⚙️ Step 2: Master Pipeline Controller & Execution Settings
# @markdown Choose whether to run the unified pipeline or target a specific module:

PIPELINE_EXECUTION_MODE = "Full End-to-End Pipeline (Screener -> Backtest -> Monte Carlo)" #@param ["Full End-to-End Pipeline (Screener -> Backtest -> Monte Carlo)", "Module 1 Only: Weekly Screener & Dynamic Watchlist", "Module 2 Only: Strategy Backtest on Custom / Scanned Watchlist", "Module 3 Only: Institutional Prop Monte Carlo & Capacity Sizing"]

# Screener Settings
universe_mode = "Combined (Indices + Broader Growth)" #@param ["Combined (Indices + Broader Growth)", "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)", "Outside Major Indices (Broader Growth & Mid-Caps)"]
top_watchlist_count = 50 #@param {type:"integer"}
render_candlestick_charts = True #@param {type:"boolean"}
max_charts_to_display = 10 #@param {type:"integer"}

# Backtest & Simulation Settings
backtest_years = 3 #@param {type:"integer"}
initial_capital = 100000.0 #@param {type:"number"}
position_sizing_mode = "Risk-Based (Equal Risk Contribution)" #@param ["Risk-Based (Equal Risk Contribution)", "Fixed Capital Allocation (3% per Trade)"]
custom_override_symbols = "" #@param {type:"string"}

print(f"🚀 Selected Execution Mode: {PIPELINE_EXECUTION_MODE}")
print(f"🌐 Universe Mode: {universe_mode} | Watchlist Target Size: {top_watchlist_count}")
print(f"💰 Account Equity: ${initial_capital:,.2f} | Backtest Horizon: {backtest_years} Years")
print(f"⚖️ Sizing Mode: {position_sizing_mode}")
if custom_override_symbols.strip():
    print(f"✏️ Custom Symbol Override Active: {custom_override_symbols.strip()}")


In [ ]:
# @title 🧠 Step 3: Module 1 — Weekly Swing Screener & Candidate Discovery Engine

# Segmented Universes
MAJOR_INDICES_UNIVERSE = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AVGO', 'COST', 'PEP',
    'CSCO', 'ADBE', 'NFLX', 'CMCSA', 'INTC', 'QCOM', 'TXN', 'AMGN', 'HON', 'INTU',
    'BKNG', 'SBUX', 'MDLZ', 'ISRG', 'GILD', 'ADI', 'ADP', 'REGN', 'VRTX', 'PANW',
    'KLAC', 'SNPS', 'CDNS', 'MELI', 'PYPL', 'CRWD', 'MAR', 'ORLY', 'CTAS', 'NXPI',
    'LRCX', 'MNST', 'PCAR', 'FTNT', 'KDP', 'CHTR', 'AEP', 'CPRT', 'PAYX', 'ROST',
    'ODFL', 'FAST', 'GEHC', 'IDXX', 'VRSK', 'CSX', 'EXC', 'FANG', 'BKR', 'KHC',
    'XEL', 'TEAM', 'DXCM', 'CCEP', 'MCHP', 'MRVL', 'CTSH', 'BIIB', 'ANSS', 'DLTR',
    'JNJ', 'UNH', 'JPM', 'XOM', 'PG', 'HD', 'CVX', 'MRK', 'ABBV', 'KO',
    'BAC', 'PFE', 'TMO', 'WMT', 'DIS', 'MCD', 'ABT', 'WFC', 'VZ', 'DHR',
    'NEE', 'PM', 'NKE', 'LIN', 'BMY', 'UNP', 'LOW', 'AMAT', 'BA', 'COP',
    'GS', 'CAT', 'IBM', 'GE', 'RTX', 'SPGI', 'PLD', 'MS', 'BLK', 'DE',
    'SCHW', 'AXP', 'SYK', 'T', 'ELV', 'MDT', 'C', 'CI', 'TJX', 'LMT',
    'MMC', 'CB', 'V', 'MA', 'FI', 'SO', 'DUK', 'EOG', 'BSX', 'PNC',
    'ITW', 'BDX', 'CME', 'EQIX', 'SLB', 'CL', 'MO', 'WM', 'AON', 'ICE',
    'SHW', 'APD', 'ECL', 'ETN', 'MCK', 'HUM', 'NSC', 'PGR', 'EMR', 'FDX',
    'ROP', 'COF', 'HCA', 'PSA', 'GD', 'TDG', 'AIG', 'PSX', 'OXY', 'MPC',
    'MET', 'CAH', 'WST', 'EG'
]

BROADER_MARKET_GROWTH_UNIVERSE = [
    'CELH', 'SMCI', 'ARM', 'APP', 'PLTR', 'DUOL', 'IOT', 'HIMS', 'TOST', 'SYM',
    'CAVA', 'RDDT', 'ASTS', 'RKLB', 'AXON', 'HOOD', 'AFRM', 'SOFI', 'DKNG', 'PATH',
    'MDB', 'ZS', 'DDOG', 'NET', 'HUBS', 'CFLT', 'BILL', 'TWLO', 'ESTC', 'GTLB',
    'FSLR', 'ENPH', 'ON', 'MPWR', 'ALGN', 'PODD', 'TMDX', 'INSP', 'EXAS', 'NTRA',
    'FRO', 'STNG', 'INSW', 'TRMD', 'TNK', 'ECO', 'ASC', 'DHT', 'HAFN', 'GNK',
    'VLO', 'MPC', 'PSX', 'PXD', 'FANG', 'DVN', 'OVV', 'APA', 'AR', 'RRC',
    'Builders', 'BLDR', 'TREX', 'EXP', 'VMC', 'MLM', 'SUM', 'CRH', 'GVA', 'AGCO',
    'EME', 'PWR', 'FIX', 'DY', 'MYRG', 'MTZ', 'PRIM', 'J', 'FLR', 'STRL',
    'DECK', 'ONON', 'BIRK', 'SKX', 'CROX', 'LULU', 'BOOT', 'BURL', 'ANF', 'AEO',
    'WING', 'SHAK', 'TXRH', 'CAVA', 'BROS', 'SG', 'LOCO', 'EAT', 'PLAY', 'DRI',
    'TTCF', 'BRBR', 'POWI', 'CRDO', 'ACMR', 'FORM', 'CAMT', 'AOSL', 'DIOD', 'SITM'
]

def get_screener_universe(mode):
    clean_indices = list(dict.fromkeys(MAJOR_INDICES_UNIVERSE))
    clean_broader = list(dict.fromkeys([s for s in BROADER_MARKET_GROWTH_UNIVERSE if s not in clean_indices]))
    if mode == "Inside Major Indices (S&P 500, Nasdaq 100, Dow 30)":
        return [(s, "INDEX") for s in clean_indices]
    elif mode == "Outside Major Indices (Broader Growth & Mid-Caps)":
        return [(s, "BROADER_MARKET") for s in clean_broader]
    else:
        return [(s, "INDEX") for s in clean_indices] + [(s, "BROADER_MARKET") for s in clean_broader]

def calculate_screener_indicators(df):
    if len(df) < 200:
        return None
    df = df.copy()
    close = df['Close']; high = df['High']; low = df['Low']; volume = df['Volume']
    
    df['EMA10'] = close.ewm(span=10, adjust=False).mean()
    df['EMA21'] = close.ewm(span=21, adjust=False).mean()
    df['SMA50'] = close.rolling(window=50).mean()
    df['SMA150'] = close.rolling(window=150).mean()
    df['SMA200'] = close.rolling(window=200).mean()
    df['VOL_SMA100'] = volume.rolling(window=100).mean()
    df['VOL_SMA20'] = volume.rolling(window=20).mean()
    
    df['LOW_52W'] = low.rolling(window=252, min_periods=180).min()
    df['HIGH_52W'] = high.rolling(window=252, min_periods=180).max()
    df['LOW_21D'] = low.rolling(window=21).min()
    
    df['EMA21_SLOPE'] = (df['EMA21'] - df['EMA21'].shift(5)) / df['EMA21'].shift(5) * 100
    df['SMA50_SLOPE'] = (df['SMA50'] - df['SMA50'].shift(5)) / df['SMA50'].shift(5) * 100
    
    # ATR 14
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    df['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['ATR14'] = df['TR'].rolling(window=14).mean()
    
    # IBD RS 14
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['RS_SCORE'] = 100 - (100 / (1 + rs))
    return df

def find_two_swing_points(df, window=30):
    recent = df.iloc[-window:].copy()
    lows = []; highs = []
    for i in range(2, len(recent) - 2):
        l = recent['Low'].iloc[i]
        if l < recent['Low'].iloc[i-1] and l < recent['Low'].iloc[i-2] and l < recent['Low'].iloc[i+1] and l < recent['Low'].iloc[i+2]:
            lows.append((l, recent.index[i].strftime('%Y-%m-%d')))
        h = recent['High'].iloc[i]
        if h > recent['High'].iloc[i-1] and h > recent['High'].iloc[i-2] and h > recent['High'].iloc[i+1] and h > recent['High'].iloc[i+2]:
            highs.append((h, recent.index[i].strftime('%Y-%m-%d')))
            
    sl1, sl1_dt = lows[-2] if len(lows) >= 2 else (recent['Low'].min(), recent.index[0].strftime('%Y-%m-%d'))
    sl2, sl2_dt = lows[-1] if len(lows) >= 1 else (recent['Low'].iloc[-5:].min(), recent.index[-5].strftime('%Y-%m-%d'))
    sh1, sh1_dt = highs[-2] if len(highs) >= 2 else (recent['High'].max(), recent.index[0].strftime('%Y-%m-%d'))
    sh2, sh2_dt = highs[-1] if len(highs) >= 1 else (recent['High'].iloc[-5:].max(), recent.index[-5].strftime('%Y-%m-%d'))
    
    higher_low = "YES (SL2 >= SL1)" if sl2 >= sl1 else "NO (Lower Low)"
    c1 = sh1 - sl1; c2 = sh2 - sl2
    contraction = "YES (VCP)" if (c1 > 0 and c2 < c1) else "NO"
    return sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction

def evaluate_candidate(sym, df, category="INDEX"):
    last = df.iloc[-1]; prev = df.iloc[-2]
    price = float(last['Close'])
    if price < 10.0: return None
    
    vol_100k = float(last['VOL_SMA100']) / 1000.0
    if vol_100k < 250.0: return None
    
    start_yr = df.loc[df.index >= f"{df.index[-1].year}-01-01"]
    if len(start_yr) > 0:
        ytd_pct = (price - float(start_yr['Close'].iloc[0])) / float(start_yr['Close'].iloc[0]) * 100.0
    else:
        ytd_pct = 15.0
    if ytd_pct <= 8.0: return None
    
    e10 = float(last['EMA10']); e21 = float(last['EMA21'])
    s50 = float(last['SMA50']); s150 = float(last['SMA150']); s200 = float(last['SMA200'])
    
    if not (e10 >= e21 and e21 >= s50 and s50 >= s150 and s150 >= s200): return None
    
    low_52w = float(last['LOW_52W'])
    if (price - low_52w) / low_52w * 100.0 < 15.0: return None
    
    dist_s50_pct = (price - s50) / s50 * 100.0
    if not (0.0 <= dist_s50_pct <= 10.0): return None
    
    dist_s200_pct = (price - s200) / s200 * 100.0
    if dist_s200_pct < 10.0: return None
    
    vol = float(last['Volume']); vol_prev = float(prev['Volume'])
    if vol_prev > 0 and ((vol - vol_prev) / vol_prev * 100.0) <= 10.0: return None
    
    dist_e21_pct = abs(price - e21) / e21 * 100.0
    slope_e21 = float(last['EMA21_SLOPE']); slope_s50 = float(last['SMA50_SLOPE'])
    
    # Multi-timeframe Demand Matrix
    low_21d = float(last['LOW_21D'])
    daily_demand = "YES (<=3.5% from 21d low)" if (price - low_21d) / low_21d * 100.0 <= 3.5 else "NO"
    weekly_demand = "YES (Near Weekly 21 EMA Support)" if dist_e21_pct <= 2.8 else "NEUTRAL"
    monthly_demand = "YES (Above Rising 10M Baseline)" if price > s200 else "NEUTRAL"
    sell_zone = "YES (Extended >7% above 21 EMA)" if (price - e21) / e21 * 100.0 > 7.0 else "NO (Low-Risk Accumulation Area)"
    vol_tag = "YES (Green Pocket Pivot)" if (vol > float(last['VOL_SMA20']) and price >= float(prev['Close'])) else "NORMAL"
    
    sl1, sl1_dt, sl2, sl2_dt, sh1, sh1_dt, sh2, sh2_dt, higher_low, contraction = find_two_swing_points(df)
    
    # Setup Horizon Classification
    if dist_e21_pct <= 2.0 and daily_demand.startswith("YES"):
        horizon = "TIER 1: TRIGGER-READY (THIS WEEK / 0-3d)"
        buy_zone = "Dual Convergence Sweet Spot"
    elif dist_e21_pct <= 4.2:
        horizon = "TIER 2: ON-DECK (1-2 WEEKS)"
        buy_zone = "Upper Pullback Zone"
    else:
        horizon = "TIER 3: BASE FORMATION (2-3 WEEKS)"
        buy_zone = "Extended Consolidation"
        
    atr = float(last['ATR14'])
    raw_risk = max(price * 0.05, min(price * 0.10, atr * 1.5))
    stop_loss = round(price - raw_risk, 2)
    risk_pct = round((price - stop_loss) / price * 100.0, 1)
    t1 = round(price * 1.05, 2); t2 = round(price * 1.10, 2); t3 = round(price * 1.15, 2)
    rr = round(10.0 / risk_pct, 2) if risk_pct > 0 else 1.5
    
    # Institutional Quality Score (IQS, 0-100) & Grade (A+ to B-)
    iqs = 50
    if dist_e21_pct <= 1.5: iqs += 20
    elif dist_e21_pct <= 2.5: iqs += 15
    elif dist_e21_pct <= 3.5: iqs += 8
    
    if slope_e21 > 0.05: iqs += 10
    if slope_s50 > 0.03: iqs += 5
    if daily_demand.startswith("YES"): iqs += 10
    if vol_tag.startswith("YES"): iqs += 10
    if higher_low.startswith("YES"): iqs += 10
    if contraction.startswith("YES"): iqs += 5
    iqs = min(100, iqs)
    
    if iqs >= 90: grade = "A+"
    elif iqs >= 80: grade = "A"
    elif iqs >= 70: grade = "B+"
    elif iqs >= 60: grade = "B"
    else: grade = "B-"
    
    return {
        'symbol': sym, 'category': category, 'setup_horizon': horizon, 'grade': grade,
        'close': round(price, 2), 'iqs_score': iqs, 'buy_zone': buy_zone,
        'dist_e21': round(dist_e21_pct, 1), 'dist_s50': round(dist_s50_pct, 1),
        'slope_e21': round(slope_e21, 2), 'slope_s50': round(slope_s50, 2),
        'daily_demand': daily_demand, 'weekly_demand': weekly_demand, 'monthly_demand': monthly_demand,
        'sell_zone': sell_zone, 'vol_tag': vol_tag, 'higher_low': higher_low, 'contraction': contraction,
        'sl1': round(sl1, 2), 'sl1_dt': sl1_dt, 'sl2': round(sl2, 2), 'sl2_dt': sl2_dt,
        'sh1': round(sh1, 2), 'sh1_dt': sh1_dt, 'sh2': round(sh2, 2), 'sh2_dt': sh2_dt,
        'entry': round(price, 2), 'stop_loss': stop_loss, 'risk_pct': risk_pct,
        't1': t1, 't2': t2, 't3': t3, 'rr': rr, 'rs_score': round(float(last['RS_SCORE']), 1)
    }

print("✅ Screener Engine & Quant Models ready.")


In [ ]:
# @title 🚀 Step 4: Execute Screening & Build Pipeline Watchlist (Top 50 Graded A+ to B-)

PIPELINE_CANDIDATE_SYMBOLS = []
screened_results = []
chart_storage = {}

if "Module 2 Only" in PIPELINE_EXECUTION_MODE or "Module 3 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 1 Screener (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
    if custom_override_symbols.strip():
        PIPELINE_CANDIDATE_SYMBOLS = [s.strip().upper() for s in custom_override_symbols.split(',') if s.strip()]
    else:
        PIPELINE_CANDIDATE_SYMBOLS = ['NVDA', 'CELH', 'MSFT', 'AAPL', 'AMZN', 'META', 'GOOGL', 'AVGO', 'COST', 'PLTR']
    print(f"📌 Using active watchlist for downstream modules: {PIPELINE_CANDIDATE_SYMBOLS}")
else:
    universe = get_screener_universe(universe_mode)
    print(f"🔍 Screening {len(universe)} symbols across US equity market...")
    
    # Download data in efficient batches
    symbols_to_fetch = [s for s, _ in universe]
    cat_map = {s: c for s, c in universe}
    
    try:
        data = yf.download(symbols_to_fetch, period="2y", interval="1d", group_by='ticker', threads=True, progress=False)
    except Exception as e:
        print(f"Download error: {e}. Falling back to individual queries.")
        data = None

    for sym in symbols_to_fetch:
        try:
            if data is not None and sym in data:
                df_sym = data[sym].dropna()
            else:
                df_sym = yf.download(sym, period="2y", interval="1d", progress=False)
                if df_sym is not None: df_sym = df_sym.dropna()
                
            if df_sym is None or len(df_sym) < 200: continue
            
            # Handle multi-level columns if present
            if isinstance(df_sym.columns, pd.MultiIndex):
                df_sym.columns = df_sym.columns.get_level_values(0)
                
            df_ind = calculate_screener_indicators(df_sym)
            if df_ind is None: continue
            
            res = evaluate_candidate(sym, df_ind, category=cat_map.get(sym, "INDEX"))
            if res:
                screened_results.append(res)
                chart_storage[sym] = (df_ind, res)
        except Exception:
            continue

    # Sort primarily by Tier readiness, then by IQS Score
    screened_results.sort(key=lambda x: (
        0 if "TIER 1" in x['setup_horizon'] else (1 if "TIER 2" in x['setup_horizon'] else 2),
        -x['iqs_score']
    ))
    
    print(f"🎯 Screening Complete: Found {len(screened_results)} qualifying candidates meeting all 13 rules!")
    
    # Grade breakdown
    grade_counts = {}
    for r in screened_results:
        g = r['grade']
        grade_counts[g] = grade_counts.get(g, 0) + 1
    print(f"📊 Quality Distribution: " + " | ".join([f"Grade {k}: {v}" for k, v in sorted(grade_counts.items())]))
    
    top_export = screened_results[:top_watchlist_count]
    PIPELINE_CANDIDATE_SYMBOLS = [r['symbol'] for r in top_export]
    
    print(f"\n🔗 AUTOMATED PIPELINE FEED: Selected Top {len(PIPELINE_CANDIDATE_SYMBOLS)} symbols to feed directly into Backtest Engine:")
    print(f"{PIPELINE_CANDIDATE_SYMBOLS[:15]} ... (+{len(PIPELINE_CANDIDATE_SYMBOLS)-15} more)")


In [ ]:
# @title 📈 Step 5: Candlestick Visualizer (Top 10 Auto-Display + Interactive Dropdown Inspector)

if render_candlestick_charts and chart_storage:
    def plot_candlestick_chart(sym, df_ind, cand):
        sub = df_ind.iloc[-35:].copy()
        fig, ax = plt.subplots(figsize=(13, 6), facecolor='#0d1117')
        ax.set_facecolor('#161b22')
        
        width = 0.6
        up = sub[sub['Close'] >= sub['Open']]
        down = sub[sub['Close'] < sub['Open']]
        ax.bar(up.index, up['Close'] - up['Open'], width, bottom=up['Open'], color='#00e676', edgecolor='#00e676', zorder=3)
        ax.vlines(up.index, up['Low'], up['High'], color='#00e676', linewidth=1.2, zorder=2)
        ax.bar(down.index, down['Close'] - down['Open'], width, bottom=down['Open'], color='#ff5252', edgecolor='#ff5252', zorder=3)
        ax.vlines(down.index, down['Low'], down['High'], color='#ff5252', linewidth=1.2, zorder=2)
        
        ax.plot(sub.index, sub['EMA10'], color='#00e5ff', label='10 EMA', linewidth=1.4, zorder=4)
        ax.plot(sub.index, sub['EMA21'], color='#ffd700', label='21 EMA (Prime Support)', linewidth=2.0, zorder=4)
        ax.plot(sub.index, sub['SMA50'], color='#e040fb', label='50 SMA (Institutional Demand)', linewidth=1.8, zorder=4)
        if 'SMA200' in sub.columns and not pd.isna(sub['SMA200'].iloc[-1]):
            ax.plot(sub.index, sub['SMA200'], color='#ff1744', label='200 SMA (Macro Baseline)', linewidth=1.4, linestyle='--', zorder=4)
            
        low_21 = float(sub['Low'].min())
        ax.axhspan(low_21, low_21 * 1.035, color='#00e676', alpha=0.15, label='Daily Demand Zone (<=3.5% above 21d low)', zorder=1)
        
        if cand.get('sl1', 0) > 0:
            ax.scatter([sub.index[-12]], [cand['sl1']], color='#00e676', s=90, marker='^', zorder=5)
            ax.text(sub.index[-12], cand['sl1'] * 0.985, f"SL1: ${cand['sl1']:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        if cand.get('sl2', 0) > 0:
            ax.scatter([sub.index[-4]], [cand['sl2']], color='#00e676', s=110, marker='^', zorder=5)
            ax.text(sub.index[-4], cand['sl2'] * 0.985, f"SL2: ${cand['sl2']:.2f}", color='#00e676', fontsize=8, ha='center', weight='bold')
        if cand.get('sh1', 0) > 0:
            ax.scatter([sub.index[-18]], [cand['sh1']], color='#ff5252', s=90, marker='v', zorder=5)
            ax.text(sub.index[-18], cand['sh1'] * 1.01, f"SH1: ${cand['sh1']:.2f}", color='#ff5252', fontsize=8, ha='center', weight='bold')
        if cand.get('sh2', 0) > 0:
            ax.scatter([sub.index[-8]], [cand['sh2']], color='#ff5252', s=110, marker='v', zorder=5)
            ax.text(sub.index[-8], cand['sh2'] * 1.01, f"SH2: ${cand['sh2']:.2f}", color='#ff5252', fontsize=8, ha='center', weight='bold')
            
        ax.axhline(cand['entry'], color='#00e5ff', linestyle='--', linewidth=1.5, label=f"Projected Entry: ${cand['entry']:.2f}")
        ax.axhline(cand['stop_loss'], color='#ff1744', linestyle='--', linewidth=1.5, label=f"Stop Loss: ${cand['stop_loss']:.2f} (-{cand['risk_pct']}%)")
        ax.axhline(cand['t1'], color='#69f0ae', linestyle=':', linewidth=1.2, label=f"Target 1 (+5%): ${cand['t1']:.2f}")
        ax.axhline(cand['t2'], color='#ffd700', linestyle=':', linewidth=1.2, label=f"Target 2 (+10%): ${cand['t2']:.2f} (R:R {cand['rr']}:1)")
        
        ax.grid(True, color='#21262d', linestyle='--', alpha=0.6)
        ax.tick_params(colors='#8b949e', labelsize=9)
        for spine in ax.spines.values(): spine.set_color('#30363d')
        
        plt.suptitle(f"{sym} [{cand['category']}] — {cand['setup_horizon']} | Grade: {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}/100)", color='#f0f6fc', fontsize=12, fontweight='bold', y=0.98)
        plt.title(f"Buy Zone: {cand['buy_zone']} | Daily Demand: {cand['daily_demand']} | Weekly Demand: {cand['weekly_demand']} | Vol Pocket Pivot: {cand['vol_tag']}", color='#8b949e', fontsize=9, pad=8)
        ax.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9', loc='upper left', fontsize=8, ncol=2)
        plt.tight_layout()
        plt.show()
        plt.close()

    # 1. Automatically render Top 10 Charts
    top_10 = list(chart_storage.items())[:min(10, len(chart_storage))]
    print(f"📊 Displaying Top {len(top_10)} Graded Candlestick Setups:")
    for sym, (df_ind, cand) in top_10:
        plot_candlestick_chart(sym, df_ind, cand)
        
    # 2. Interactive Dropdown Setup Inspector
    dropdown_opts = [
        (f"{sym} [{cand['category']}] — Grade {cand.get('grade', 'B')} (IQS: {cand['iqs_score']}) | {cand['setup_horizon'][:18]}", sym)
        for sym, (df_ind, cand) in chart_storage.items()
    ]
    
    if dropdown_opts:
        print("\n" + "="*80)
        print("🎯 INTERACTIVE SETUP EXPLORER: Select any candidate ticker from the dropdown below to load its chart:")
        print("="*80)
        
        dd = widgets.Dropdown(
            options=dropdown_opts,
            value=dropdown_opts[0][1],
            description='🔍 Select Ticker:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='600px')
        )
        
        btn = widgets.Button(
            description='Load Selected Chart',
            button_style='primary',
            icon='search',
            layout=widgets.Layout(width='180px')
        )
        
        chart_box = widgets.Output()
        
        def render_dd_chart(target_sym):
            with chart_box:
                clear_output(wait=True)
                if target_sym in chart_storage:
                    d_df, d_cand = chart_storage[target_sym]
                    print(f"⚡ Rendering chart for: {target_sym}")
                    plot_candlestick_chart(target_sym, d_df, d_cand)
                    
        def on_change(change):
            if change.get('type') == 'change' and change.get('name') == 'value':
                render_dd_chart(change['new'])
                
        def on_click(b):
            render_dd_chart(dd.value)
            
        dd.observe(on_change, names='value')
        btn.on_click(on_click)
        
        display(widgets.HBox([dd, btn]))
        display(chart_box)
else:
    print("Chart visualizer skipped or no candidates available.")


In [ ]:
# @title 📊 Step 6: Module 2 — Swing Strategy Backtester (Driven by Screener Watchlist)

PIPELINE_TRADE_LOG = []
strategy_metrics = {}

if "Module 1 Only" in PIPELINE_EXECUTION_MODE or "Module 3 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 2 Backtester (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
else:
    target_symbols = PIPELINE_CANDIDATE_SYMBOLS[:20] if len(PIPELINE_CANDIDATE_SYMBOLS) > 0 else ['NVDA', 'CELH', 'MSFT', 'AAPL', 'AMZN']
    print(f"⚡ Running Multi-Year Vectorized Swing Backtest on {len(target_symbols)} symbols: {target_symbols}")
    
    start_date = (datetime.now() - timedelta(days=backtest_years*365)).strftime('%Y-%m-%d')
    end_date = datetime.now().strftime('%Y-%m-%d')
    
    trades = []
    equity = initial_capital
    equity_curve = [equity]
    
    for sym in target_symbols:
        try:
            df = yf.download(sym, start=start_date, end=end_date, interval="1d", progress=False)
            if df is None or len(df) < 60: continue
            if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
            
            c = df['Close']; h = df['High']; l = df['Low']
            ema21 = c.ewm(span=21, adjust=False).mean()
            sma50 = c.rolling(50).mean()
            tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
            atr14 = tr.rolling(14).mean()
            
            in_pos = False
            entry_p = 0.0; sl_p = 0.0; shares = 0; entry_dt = None
            
            for i in range(50, len(df)):
                price = float(c.iloc[i])
                dt = df.index[i].strftime('%Y-%m-%d')
                
                if not in_pos:
                    # Gate 2 Pullback condition: price near rising 21 EMA / 50 SMA
                    dist_ema = abs(price - float(ema21.iloc[i])) / float(ema21.iloc[i]) * 100.0
                    if dist_ema <= 2.5 and price > float(sma50.iloc[i]) and float(ema21.iloc[i]) > float(sma50.iloc[i]):
                        in_pos = True
                        entry_p = price
                        entry_dt = dt
                        atr_val = float(atr14.iloc[i]) if not pd.isna(atr14.iloc[i]) else price * 0.04
                        risk_amt = max(entry_p * 0.05, min(entry_p * 0.10, atr_val * 1.5))
                        sl_p = entry_p - risk_amt
                        
                        # Position sizing
                        if "Risk-Based" in position_sizing_mode:
                            risk_per_share = entry_p - sl_p
                            dollar_risk = equity * 0.02 # 2% account risk
                            shares = max(1, int(dollar_risk / risk_per_share)) if risk_per_share > 0 else int(equity * 0.05 / entry_p)
                        else:
                            shares = max(1, int((equity * 0.03) / entry_p)) # Fixed 3% capital
                else:
                    high_p = float(h.iloc[i])
                    low_p = float(l.iloc[i])
                    
                    # Target 1 (+5%) or Target 2 (+10%) or Stop Loss
                    if low_p <= sl_p:
                        pnl = (sl_p - entry_p) * shares
                        trades.append({
                            'symbol': sym, 'entry_date': entry_dt, 'exit_date': dt,
                            'entry_price': entry_p, 'exit_price': sl_p, 'shares': shares,
                            'pnl': pnl, 'return_pct': (sl_p - entry_p) / entry_p * 100.0,
                            'exit_reason': 'STOP_LOSS'
                        })
                        equity += pnl
                        equity_curve.append(equity)
                        in_pos = False
                    elif high_p >= entry_p * 1.10: # Hit Target 2
                        exit_price = entry_p * 1.10
                        pnl = (exit_price - entry_p) * shares
                        trades.append({
                            'symbol': sym, 'entry_date': entry_dt, 'exit_date': dt,
                            'entry_price': entry_p, 'exit_price': exit_price, 'shares': shares,
                            'pnl': pnl, 'return_pct': 10.0,
                            'exit_reason': 'TARGET_2_HIT'
                        })
                        equity += pnl
                        equity_curve.append(equity)
                        in_pos = False
        except Exception:
            continue

    PIPELINE_TRADE_LOG = trades
    df_trades = pd.DataFrame(trades)
    
    if len(df_trades) > 0:
        wins = df_trades[df_trades['pnl'] > 0]
        losses = df_trades[df_trades['pnl'] <= 0]
        win_rate = len(wins) / len(df_trades) * 100.0
        tot_profit = wins['pnl'].sum() if len(wins) > 0 else 0
        tot_loss = abs(losses['pnl'].sum()) if len(losses) > 0 else 1.0
        pf = tot_profit / tot_loss if tot_loss > 0 else 9.99
        tot_return = (equity - initial_capital) / initial_capital * 100.0
        
        # Max Drawdown
        eq_series = pd.Series(equity_curve)
        peak = eq_series.cummax()
        dd = (eq_series - peak) / peak * 100.0
        max_dd = abs(dd.min())
        
        strategy_metrics = {
            'total_trades': len(df_trades), 'win_rate': round(win_rate, 1),
            'profit_factor': round(pf, 2), 'total_return_pct': round(tot_return, 1),
            'max_drawdown_pct': round(max_dd, 1), 'final_equity': round(equity, 2)
        }
        
        print("\n" + "="*80)
        print("🏛️ EMPIRICAL BACKTEST PERFORMANCE SUMMARY (Driven by Screener Setups):")
        print("="*80)
        print(f"• Total Executed Trades: {strategy_metrics['total_trades']}")
        print(f"• Win Rate: {strategy_metrics['win_rate']}%")
        print(f"• Profit Factor: {strategy_metrics['profit_factor']}")
        print(f"• Total Portfolio Return: +{strategy_metrics['total_return_pct']}%")
        print(f"• Max Drawdown: -{strategy_metrics['max_drawdown_pct']}%")
        print(f"• Ending Account Equity: ${strategy_metrics['final_equity']:,.2f}")
        print("="*80)
        
        # Plot Equity Curve
        plt.figure(figsize=(12, 4), facecolor='#0d1117')
        ax = plt.gca()
        ax.set_facecolor('#161b22')
        plt.plot(equity_curve, color='#00e676', linewidth=2.0, label='Portfolio Equity ($)')
        plt.title(f"Rajat Alpha V67 Equity Growth Curve (Screener Watchlist Backtest) | Return: +{tot_return:.1f}%", color='#f0f6fc', fontsize=12, fontweight='bold')
        plt.xlabel('Closed Trade #', color='#8b949e')
        plt.ylabel('Account Equity ($)', color='#8b949e')
        plt.grid(True, color='#21262d', linestyle='--', alpha=0.6)
        plt.tick_params(colors='#8b949e')
        for spine in ax.spines.values(): spine.set_color('#30363d')
        plt.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
        plt.tight_layout()
        plt.show()
        plt.close()
    else:
        print("No simulated trades generated in backtest period.")


In [ ]:
# @title 🎲 Step 7: Module 3 — Institutional Prop Firm Risk & Monte Carlo Protocol

if "Module 1 Only" in PIPELINE_EXECUTION_MODE:
    print(f"⏩ Skipping Module 3 Monte Carlo (Execution Mode is set to {PIPELINE_EXECUTION_MODE})")
else:
    # Use metrics from Module 2 if available; otherwise use robust strategy benchmarks
    p_win = (strategy_metrics.get('win_rate', 77.8) / 100.0)
    profit_factor = strategy_metrics.get('profit_factor', 3.68)
    avg_win_dollar = 450.0
    avg_loss_dollar = avg_win_dollar / max(1.0, (profit_factor * (1.0 - p_win) / p_win))
    
    print("🎲 Executing 10,000-Path Monte Carlo Simulation for Prop Challenge & Risk of Ruin...")
    print(f"• Baseline Win Rate: {p_win*100:.1f}% | Profit Factor: {profit_factor:.2f}")
    
    NUM_SIMS = 10000
    TRADES_PER_SIM = 60
    TARGET_PROFIT = initial_capital * 0.10 # +10% target
    MAX_PERMITTED_DD = initial_capital * 0.08 # -8% max drawdown
    
    np.random.seed(42)
    win_outcomes = np.random.binomial(1, p_win, size=(NUM_SIMS, TRADES_PER_SIM))
    
    passed_challenge = 0
    breached_drawdown = 0
    final_equities = []
    
    for sim_idx in range(NUM_SIMS):
        cur_eq = initial_capital
        peak_eq = initial_capital
        breached = False
        passed = False
        
        for trade_idx in range(TRADES_PER_SIM):
            is_win = win_outcomes[sim_idx, trade_idx]
            if is_win:
                pnl = np.random.normal(avg_win_dollar, avg_win_dollar * 0.2)
            else:
                pnl = -abs(np.random.normal(avg_loss_dollar, avg_loss_dollar * 0.15))
                
            cur_eq += pnl
            if cur_eq > peak_eq: peak_eq = cur_eq
            
            dd = peak_eq - cur_eq
            if dd >= MAX_PERMITTED_DD:
                breached = True
                
            if cur_eq >= initial_capital + TARGET_PROFIT and not breached:
                passed = True
                
        if passed and not breached: passed_challenge += 1
        if breached: breached_drawdown += 1
        final_equities.append(cur_eq)
        
    prob_pass_single = (passed_challenge / NUM_SIMS) * 100.0
    prob_dd_breach = (breached_drawdown / NUM_SIMS) * 100.0
    prob_pass_4_attempts = (1.0 - (1.0 - (prob_pass_single / 100.0))**4) * 100.0
    
    print("\n" + "="*80)
    print("🏛️ INSTITUTIONAL PROP PROTOCOL MONTE CARLO STRESS TEST RESULTS:")
    print("="*80)
    print(f"• Single-Attempt Pass Probability (+10% Profit Target): {prob_pass_single:.2f}%")
    print(f"• Cumulative 4-Attempt Pass Probability (Multi-Stage): {prob_pass_4_attempts:.2f}%")
    print(f"• Probability of Hitting Max Permitted Drawdown (-8% DD): {prob_dd_breach:.2f}%")
    print(f"• Mean Ending Portfolio Equity: ${np.mean(final_equities):,.2f}")
    print(f"• 5th Percentile Conservative Equity Floor: ${np.percentile(final_equities, 5):,.2f}")
    print("="*80)
    
    # Plot Monte Carlo Distribution
    plt.figure(figsize=(12, 4), facecolor='#0d1117')
    ax = plt.gca()
    ax.set_facecolor('#161b22')
    plt.hist(final_equities, bins=60, color='#00e5ff', edgecolor='#161b22', alpha=0.85)
    plt.axvline(initial_capital + TARGET_PROFIT, color='#00e676', linestyle='--', linewidth=2.0, label=f'Target (+10%): ${initial_capital+TARGET_PROFIT:,.0f}')
    plt.axvline(initial_capital - MAX_PERMITTED_DD, color='#ff1744', linestyle='--', linewidth=2.0, label=f'Max DD (-8%): ${initial_capital-MAX_PERMITTED_DD:,.0f}')
    plt.axvline(initial_capital, color='#ffd700', linestyle=':', linewidth=1.5, label='Break-Even')
    plt.title(f"10,000-Path Monte Carlo Terminal Equity Distribution | Cumulative Pass: {prob_pass_4_attempts:.1f}%", color='#f0f6fc', fontsize=12, fontweight='bold')
    plt.xlabel('Simulated Ending Equity ($)', color='#8b949e')
    plt.ylabel('Simulation Frequency', color='#8b949e')
    plt.grid(True, color='#21262d', linestyle='--', alpha=0.6)
    plt.tick_params(colors='#8b949e')
    for spine in ax.spines.values(): spine.set_color('#30363d')
    plt.legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
# @title 💾 Step 8: Module 4 — Master Export & 1-Click Download Center

import os
from google.colab import files

print("📦 Packaging Institutional Artifacts for Download...")

# 1. Export watchlist.txt
output_txt_file = "watchlist.txt"
with open(output_txt_file, "w", encoding="utf-8") as f:
    for sym in PIPELINE_CANDIDATE_SYMBOLS:
        f.write(f"{sym}\n")
print(f"✅ Generated: {output_txt_file} ({len(PIPELINE_CANDIDATE_SYMBOLS)} symbols ready for Single_Buy2.0)")

# 2. Export 6-Sheet Excel Screener Workbook if candidates exist
excel_out = "weekly_swing_screener_results.xlsx"
if screened_results:
    wb = openpyxl.Workbook()
    ws_default = wb.active
    wb.remove(ws_default)
    
    navy_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    regular_font = Font(name="Calibri", size=10)
    
    headers = [
        "Rank", "Symbol", "Category", "Setup Horizon", "Price ($)", "IQS Score", "Grade",
        "Buy Zone", "Dist EMA21 (%)", "Dist SMA50 (%)", "Daily Demand", "Weekly Demand",
        "Volume Pocket Pivot", "Higher Low", "VCP", "SL1 ($)", "SL2 ($)", "SH1 ($)", "SH2 ($)",
        "Entry ($)", "Stop Loss ($)", "Risk (%)", "Target 1 (+5%)", "Target 2 (+10%)", "Risk/Reward"
    ]
    
    def populate(ws, cand_list):
        ws.append(headers)
        for col in range(1, len(headers)+1):
            cell = ws.cell(row=1, column=col)
            cell.fill = navy_fill; cell.font = header_font; cell.alignment = Alignment(horizontal="center")
            
        for idx, r in enumerate(cand_list, 1):
            ws.append([
                idx, r['symbol'], r['category'], r['setup_horizon'], r['close'], r['iqs_score'], r.get('grade', 'B'),
                r['buy_zone'], r['dist_e21'], r['dist_s50'], r['daily_demand'], r['weekly_demand'],
                r['vol_tag'], r['higher_low'], r['contraction'], r['sl1'], r['sl2'], r['sh1'], r['sh2'],
                r['entry'], r['stop_loss'], r['risk_pct'], r['t1'], r['t2'], r['rr']
            ])
            for c_idx in range(1, len(headers)+1):
                c = ws.cell(row=ws.max_row, column=c_idx)
                c.font = regular_font
                c.alignment = Alignment(horizontal="center" if c_idx in [1, 2, 3, 4, 7, 11, 12, 13, 14, 15] else "right")
                
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_len + 3, 12)

    # Sheet 1: Index Top 50
    ws1 = wb.create_sheet("Index_Watchlist_Top_Setups")
    populate(ws1, [c for c in screened_results if c['category'] == "INDEX"][:top_watchlist_count])
    
    # Sheet 2: Broader Market Alpha Top 50
    ws2 = wb.create_sheet("Broader_Market_Alpha_Setups")
    populate(ws2, [c for c in screened_results if c['category'] == "BROADER_MARKET"][:top_watchlist_count])
    
    # Sheet 3: Trigger Ready This Week
    ws3 = wb.create_sheet("Trigger_Ready_This_Week")
    populate(ws3, [c for c in screened_results if "TIER 1" in c['setup_horizon']])
    
    # Sheet 4: TXT Export
    ws4 = wb.create_sheet("Watchlist_TXT_Export")
    ws4.append(["Symbol", "Category", "Setup Horizon", "IQS Score", "Grade"])
    for col in range(1, 6): ws4.cell(row=1, column=col).fill = navy_fill; ws4.cell(row=1, column=col).font = header_font
    for r in screened_results[:top_watchlist_count]:
        ws4.append([r['symbol'], r['category'], r['setup_horizon'], r['iqs_score'], r.get('grade', 'B')])
        
    wb.save(excel_out)
    print(f"✅ Generated: {excel_out} (Multi-sheet institutional workbook)")

# 3. Export backtest trade log if available
if PIPELINE_TRADE_LOG:
    pd.DataFrame(PIPELINE_TRADE_LOG).to_csv("pipeline_trade_log.csv", index=False)
    print("✅ Generated: pipeline_trade_log.csv (Complete empirical trade log)")

# Trigger downloads
try:
    if os.path.exists(output_txt_file):
        files.download(output_txt_file)
    if os.path.exists(excel_out):
        files.download(excel_out)
    if os.path.exists("pipeline_trade_log.csv"):
        files.download("pipeline_trade_log.csv")
    print("📥 Browser download triggered for all generated artifacts!")
except Exception as e:
    print(f"Note on download: {e}. Files are saved in local runtime environment.")
